# Module 7.3: FlashAttention

Welcome to the silicon level of LLM Engineering! We have successfully mapped the algebra and calculus that allows a Transformer to learn.

However, we face one final boundary: physics. 
Standard Multi-Head Attention gets incredibly slow and memory-hungry as sequences get longer. In this notebook, we look at **FlashAttention**, the groundbreaking algorithm that bypassed physical hardware limits by mathematically rethinking *how* memory is read.

## 1. The GPU Memory Wall

To understand FlashAttention, we have to look inside the GPU:
1. **HBM (High Bandwidth Memory)**: The "main memory" of the GPU (e.g. 80 GB on an A100). It holds the model weights and the attention matrices. *Large, but comparatively slow to read and write.*
2. **SRAM (on-chip memory)**: Tiny memory right next to the compute cores (e.g. ~20 MB). *Small, but very fast.*

### 👨‍🍳 The Chef Analogy
You are a chef executing a recipe (the GPU compute cores).
- Your cutting board is tiny but right in front of you (**SRAM**).
- The walk-in fridge is down the hall (**HBM**).

In **standard attention**, computing $\text{Attention} = \text{Softmax}(Q \cdot K^T) \cdot V$:
1. Read $Q$ and $K$ from the fridge.
2. Compute $Q \cdot K^T$ on the cutting board.
3. Walk back to the fridge to save this big new matrix ($N \times N$).
4. Walk back, read the matrix, apply the softmax.
5. Walk back, save the softmax matrix.
6. Walk back, read the softmax matrix and the $V$ matrix.
7. Finally multiply them.

A large fraction of the time goes to walking to the fridge rather than to actual cooking. (The fridge-walking framing is an illustration of the bottleneck, not a measured number; the precise share depends on the hardware and sequence length.)

## 2. HOW FlashAttention Works (Tiling)

FlashAttention keeps the chef at the cutting board.
Instead of loading whole matrices, it slices $Q$, $K$, and $V$ into small **tiles** (blocks) that fit in SRAM.

It loads a tile of $Q$, $K$, and $V$ at once, computes the dot products, incrementally updates a softmax normalizer without needing the full row, multiplies by $V$, and writes only the finished output back to HBM.

**Result**: it never materializes the full $N \times N$ attention matrix in HBM. By removing those $O(N^2)$ reads and writes to slow memory, it speeds attention up substantially (often around 2–4x in practice). Note the bottleneck doesn't vanish entirely; attention is still bandwidth-sensitive, FlashAttention just moves much less data.

## 3. Simulating FlashAttention in PyTorch

True FlashAttention requires writing custom `C++/CUDA` or `Triton` code to explicitly tell the GPU registers where to store data. We cannot easily do that in a pure Python notebook.

However, we CAN perfectly simulate the **Tiling Mathematics** in Python using block matrices to prove that incrementally calculating Softmax blocks yields the exact same theoretical result as Standard Attention!

In [ ]:
import torch
import torch.nn.functional as F
torch.manual_seed(0)

def standard_attention(Q, K, V):
    """
    The standard memory-hogging approach we learned in Module 2!
    Materializes the full N x N score matrix.
    """
    scale = Q.size(-1) ** 0.5
    scores = torch.matmul(Q, K.transpose(-2, -1)) / scale
    attn_weights = F.softmax(scores, dim=-1)
    return torch.matmul(attn_weights, V)

def flash_attention_simulation(Q, K, V, block_size=2):
    """
    Simulates the tiling math of FlashAttention.
    Python loops are slow, but the same math on-chip in SRAM is fast.

    Loop order note:
    The real FlashAttention forward pass loops Q blocks on the OUTER loop and K/V blocks
    on the INNER loop, so each query row's running softmax state lives in fast SRAM for the
    duration of its inner loop. Here, for simplicity, we do the OPPOSITE nesting: K/V blocks
    outer (j), Q blocks inner (i). Because a given Q block is revisited once per K block, its
    running softmax state (max `m`, denominator `l`, and partial output `O`) must SURVIVE
    across all K blocks. That is exactly why m, l, and O below are GLOBAL per-row arrays
    rather than locals scoped to one Q block: we keep accumulating into them every time we
    come back to the same Q rows with a new chunk of keys.
    """
    seq_len = Q.size(0)
    d_model = Q.size(-1)
    scale = d_model ** 0.5

    # This simple simulation assumes the sequence divides evenly into blocks.
    assert seq_len % block_size == 0, "This sim assumes seq_len % block_size == 0"

    # Per-row running state. In a real kernel these are the SMALL per-row trackers that live
    # in fast SRAM while a query block is being processed (NOT the big matrices in HBM).
    # The only thing eventually written back to HBM is the finished output O.
    O = torch.zeros((seq_len, d_model))          # running (unscaled) output per query row
    l = torch.zeros((seq_len, 1))                # running softmax denominator per query row
    m = torch.full((seq_len, 1), -float('inf'))  # running max per query row (for stability)

    # Tiling loops! Break the sequence into digestible blocks.
    for j in range(0, seq_len, block_size):
        K_block = K[j:j+block_size, :]  # Fetch just a tile of K
        V_block = V[j:j+block_size, :]  # Fetch just a tile of V

        for i in range(0, seq_len, block_size):
            Q_block = Q[i:i+block_size, :]  # Fetch just a tile of Q

            # Local block scores (Q * K^T)
            S_block = torch.matmul(Q_block, K_block.transpose(-2, -1)) / scale

            # --- The Online Softmax Trick ---
            # Track running maximums so exponentials never blow up.
            m_block_old = m[i:i+block_size]
            m_block_new = torch.maximum(m_block_old, torch.max(S_block, dim=-1, keepdim=True).values)

            # Local probabilities, shifted by the new running max.
            P_block = torch.exp(S_block - m_block_new)

            # Rescale the old running totals to the new max, then add this block's contribution.
            l_block_old = l[i:i+block_size]
            correction = torch.exp(m_block_old - m_block_new)
            l_block_new = correction * l_block_old + torch.sum(P_block, dim=-1, keepdim=True)

            O_block_old = O[i:i+block_size]
            O_block_new = correction * O_block_old + torch.matmul(P_block, V_block)

            # Persist the updated per-row state (must survive to the next K block).
            l[i:i+block_size] = l_block_new
            m[i:i+block_size] = m_block_new
            O[i:i+block_size] = O_block_new

    # Final pass: divide by the tracked softmax denominator.
    return O / l

print("Logic Initialized!")

In [ ]:
# TEST TIME!
# Let's create a tiny dummy sequence to test the math.
seq_length = 8
dim = 16
torch.manual_seed(42)
Q = torch.randn(seq_length, dim)
K = torch.randn(seq_length, dim)
V = torch.randn(seq_length, dim)

# 1. Run Standard Attention
standard_out = standard_attention(Q, K, V)

# 2. Run Flash Attention (Mathematically tracking it dynamically into blocks)
flash_out = flash_attention_simulation(Q, K, V, block_size=4)

# Are they identical?
# We use torch.allclose because floating point math differs slightly when processed iteratively!
is_exact_match = torch.allclose(standard_out, flash_out, atol=1e-6)

print(f"Do the simulated Tile Blocks equal the massive Standard Matrix exactly? {is_exact_match}!!")
print(f"\nSample of Standard Output:\n{standard_out[0][:4]}")
print(f"\nSample of Flash Output:\n{flash_out[0][:4]}")

## Summary

By slicing sequences into **tiles**, keeping them in fast SRAM, and tracking the softmax max and denominator online, FlashAttention avoids ever writing the full $N \times N$ attention matrix to slow HBM.

Standard attention uses $O(N^2)$ memory because it materializes that $N \times N$ score matrix. FlashAttention keeps memory at $O(N)$ by removing those quadratic reads and writes to HBM. That reduction in slow-memory traffic is a big part of why context windows have grown from a few thousand tokens toward a million or more.

### 🏋️ Try it yourself

1. **Different block sizes.** The test used `block_size=4`. Re-run `flash_attention_simulation` with `block_size=1`, `2`, and `8` (all divide 8 evenly) and confirm each still matches `standard_attention` via `torch.allclose`. The block size changes the schedule, not the result.
2. **Why the running max matters.** Make a copy of the function with the online-softmax correction removed (set `m_block_new = m_block_old` and drop the `correction` term, i.e. assume a fixed max of 0). Feed it inputs with large score values (e.g. multiply `Q` by 20) and watch the result diverge from `standard_attention`, showing why the running max is needed for numerical stability.

In [ ]:
# Your code here!
# Hint for task 1:
# for bs in [1, 2, 8]:
#     out = flash_attention_simulation(Q, K, V, block_size=bs)
#     print(bs, torch.allclose(standard_attention(Q, K, V), out, atol=1e-6))